# Synthetic Difference-in-Differences: California Proposition 99

Replicates the R `synthdid` starter code in Python:

```r
library(synthdid)
data('california_prop99')
setup = panel.matrices(california_prop99)
tau.hat = synthdid_estimate(setup$Y, setup$N0, setup$T0)
se = sqrt(vcov(tau.hat, method='placebo'))
sprintf('point estimate: %1.2f', tau.hat)
sprintf('95%% CI (%1.2f, %1.2f)', tau.hat - 1.96 * se, tau.hat + 1.96 * se)
plot(tau.hat)
```

**Dataset:** California Prop 99 (tobacco tax, 1989). Outcome is cigarette consumption (packs per capita). Treatment effect should be approximately -15 to -16 packs per capita.

## 1. Setup

In [ ]:
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.join(os.getcwd(), ".."))

from synthdid import synthdid_estimate, panel_matrices, SynthDID

## 2. Load data

## 3. Functional API

`synthdid_estimate()` returns a `SynthdidEstimate` object. All analysis methods live directly on it — no need to import separate functions.

In [ ]:
DATA_PATH = os.path.join(os.getcwd(), "..", "data", "california_prop99.csv")
df = pd.read_csv(DATA_PATH, sep=None, engine="python")

setup = panel_matrices(df, unit="State", time="Year",
                       outcome="PacksPerCapita", treatment="treated")

est = synthdid_estimate(
    setup["Y"], setup["N0"], setup["T0"],
    unit_names=setup["unit_names"],
    time_names=[str(t) for t in setup["time_names"]],
)
est

### Results summary

`.summary()` computes the SE via the placebo method and returns a formatted table.

In [ ]:
results = est.summary(se_method="placebo", replications=200)
print(results)

### Plot

In [ ]:
fig = est.plot(se=results.se, figsize=(12, 7))
plt.show()

### Weight importance

In [ ]:
est.top_weights(top_n=10, weight_type="omega").round(4)

In [ ]:
est.top_weights(top_n=5, weight_type="lambda").round(4)

In [ ]:
fig = est.weights_plot(top_n=10)
plt.show()

### Period-by-period effect curve

In [ ]:
detail = est.effect_curve(detail=True)
import pandas as pd
pd.DataFrame({
    "actual":    detail.actual.round(2),
    "predicted": detail.predicted.round(2),
    "tau(t)":    detail.tau.round(2),
}, index=detail.time_names)

## 4. Sklearn-style API

`SynthDID` accepts a DataFrame directly — no need to call `panel_matrices` manually. All the same methods are available via `model.result_`.

In [ ]:
model = SynthDID(se_method="placebo", replications=200)
model.fit(df, unit="State", time="Year",
          outcome="PacksPerCapita", treatment="treated")

print(model.summary())